In [ ]:
import json
import requests
from getpass import getpass
import time
import os
import pandas as pd
import random
from collections import defaultdict
import threading
from datetime import datetime
import math
import re

In [ ]:
# Send http request
def sendRequest(url, data, apiKey=None, exitIfNoResponse=True):
    """
    Send a request to an M2M (Machine-to-Machine) endpoint and return the parsed JSON response.

    Parameters:
    - url (str): The URL of the M2M endpoint.
    - data (dict): The payload to be sent with the request.
    - apiKey (str, optional): An optional API key for authorization. If not provided, the request will be sent without an authorization header.
    - exitIfNoResponse (bool, optional): If True, the program will exit upon receiving an error or no response. Defaults to True.

    Returns:
    - dict: The parsed JSON response containing the data, or False if there was an error.
    """

    # Convert payload to json string
    json_data = json.dumps(data)

    if apiKey == None:
        response = requests.post(url, json_data)
    else:
        headers = {"X-Auth-Token": apiKey}
        response = requests.post(url, json_data, headers=headers)

    try:
        httpStatusCode = response.status_code
        if response == None:
            print("No output from service")
            if exitIfNoResponse:
                return
            else:
                return False
        output = json.loads(response.text)
        if output["errorCode"] != None:
            print(output["errorCode"], "- ", output["errorMessage"])
            if exitIfNoResponse:
                return
            else:
                return False
        if httpStatusCode == 404:
            print("404 Not Found")
            if exitIfNoResponse:
                return
            else:
                return False
        elif httpStatusCode == 401:
            print("401 Unauthorized")
            if exitIfNoResponse:
                return
            else:
                return False
        elif httpStatusCode == 400:
            print("Error Code", httpStatusCode)
            if exitIfNoResponse:
                return
            else:
                return False
    except Exception as e:
        response.close()
        print(e)
        if exitIfNoResponse:
            return
        else:
            return False
    response.close()

    return output["data"]

In [ ]:
def prompt_ERS_login(serviceUrl):
    print("Logging in...\n")

    p = ["Enter EROS Registration System (ERS) Username: ", "Enter ERS Account Token: "]

    # Use requests.post() to make the login request
    response = requests.post(
        f"{serviceUrl}login-token",
        json={"username": getpass(prompt=p[0]), "token": getpass(prompt=p[1])},
    )

    # Check for successful response
    if response.status_code == 200:
        apiKey = response.json()["data"]
        print("\nLogin Successful, API Key Received!")
        headers = {"X-Auth-Token": apiKey}
        return apiKey
    else:
        print(
            "\nLogin was unsuccessful, please try again or create an account at: https://ers.cr.usgs.gov/register."
        )

In [ ]:
ATTEMPT_DICT = defaultdict(
    lambda: {
        "attempts": 0,  # how many times we've tried this URL
        "filename": None,  # filename
        "downloadId": None,  # downloadId returned from a download-request or download-retrieve
        "complete": False,  # whether the download is complete
    }
)

In [ ]:
maxthreads = 5  # Threads count for downloads
sema = threading.Semaphore(value=maxthreads)
threads = []

In [ ]:
def downloadFile(download_dict, out_dir, max_attempts=5):
    """
    Download a single file with retry control and basic bookkeeping.

    Parameters:
        - download_dict (dict): JSON dictionary returned from submitting a
                                download-request or download-retrieve and
                                should contain at least a  {'url', 'downloadId'}
                                for the file.
        - out_dir (str): Directory to write the downloaded file into.
        - max_attempts (int): Maximum retry attempts before giving up.

    Returns:
        - dict: ATTEMPT_DICT (tracks attempts and filename per URL).
    """

    download_url = download_dict["url"]
    sema.acquire()

    try:
        current_attempt = ATTEMPT_DICT[download_url]["attempts"] + 1
        ATTEMPT_DICT[download_url]["attempts"] = current_attempt
        ATTEMPT_DICT[download_url]["downloadId"] = download_dict["downloadId"]

        # Use requests to initiate download
        downloadResponse = requests.get(download_url, stream=True, timeout=5)

        # parse the filename from the Content-Disposition header
        content_disposition = downloadResponse.headers.get(
            "Content-Disposition"
        ) or downloadResponse.headers.get("content-disposition")

        # Parse the header string correctly
        filename = re.findall("filename=(.+)", content_disposition)[0].strip('"')
        ATTEMPT_DICT[download_url]["filename"] = filename
        filepath = os.path.join(out_dir, filename)

        # write the file to the destination directory
        print(f"      > Downloading: {download_url}")
        with open(filepath, "wb") as f:
            for data in downloadResponse.iter_content(chunk_size=8192):
                f.write(data)

        # Reset the counter on success
        # ATTEMPT_DICT.pop(download_url, None)

        # Set complete status
        print(f"      ✓ Complete: {filepath}")
        ATTEMPT_DICT[download_url]["complete"] = True

    except Exception as e:

        current_attempt = ATTEMPT_DICT[download_url]["attempts"] + 1
        print(
            f"      ! Attempt [{str(current_attempt)}] failed for {download_url}: {e}"
        )

        if current_attempt >= max_attempts:
            print(
                f"      !!! Stopping retries after [{str(max_attempts)}] attempts: {download_url}"
            )
            return

        # Log attempt and pause before trying again
        ATTEMPT_DICT[download_url]["attempts"] = current_attempt
        pause = math.ceil(random.uniform(5, 30))
        print(f"      !! Retrying in {pause}s {download_url}…")
        time.sleep(pause)

        # Try downloading again
        runDownload(threads, download_dict, out_dir)

    finally:
        # release semaphore
        sema.release()

    return ATTEMPT_DICT


def runDownload(threads, download_dict, out_dir):
    thread = threading.Thread(
        target=downloadFile,
        args=(
            download_dict,
            out_dir,
        ),
    )
    threads.append(thread)
    thread.start()

In [ ]:
serviceUrl = "https://m2m.cr.usgs.gov/api/api/json/stable/"

In [ ]:
apiKey = prompt_ERS_login(serviceUrl)

In [ ]:
datasetName = "landsat_ot_c2_l2"

In [ ]:
query = {
    "datasetName": datasetName,
    "maxResults": 10,
    "spatialFilter": {
        "filterType": "oli",
        "lowerLeft": {"latitude": 67.45, "longitude": -72.55},
        "upperRight": {"latitude": 67.55, "longitude": -72.45},
    },
    "temporalFilter": {"start": "2012-01-01", "end": "2012-01-05"},
    "sceneFilter": {
        "cloudCoverFilter": {"min": 0, "max": 5},
    },
}
print(query)

In [ ]:
scene_results = sendRequest(serviceUrl + "scene-search", query, apiKey)

In [ ]:
scene_df = pd.json_normalize(scene_results["results"])
scene_df.head(10)

In [ ]:
num_to_dl = 2
sceneIds = []
displayIds = []
for result in scene_results["results"]:
    # Add this scene to the list I would like to download
    sceneIds.append(result["entityId"])
    displayIds.append(result["displayId"])

sceneIds, displayIds

In [ ]:
serviceUrl

In [ ]:
download_payload = {"datasetName": datasetName, "entityIds": sceneIds}
print(download_payload)

downloadOptions = sendRequest(serviceUrl + "download-options", download_payload, apiKey)

In [ ]:
headers = {"X-Auth-Token": apiKey}
response = requests.post(serviceUrl + "download-options", data=json.dumps(download_payload), headers=headers)

In [ ]:
response.text.split("\n")

In [ ]:
availableproducts = []
for product in downloadOptions:
    # Make sure the product is available for this scene
    if product["available"] == True and product["downloadSystem"] != "folder":
        availableproducts.append(
            {"entityId": product["entityId"], "productId": product["id"]}
        )
print(f"Available products for download: {len(availableproducts)}")

In [ ]:
requestedDownloadsCount = len(availableproducts)

# set a label for the download request
label = datetime.now().strftime("%Y%m%d_%H%M%S")  # Customized label using date time
download_req_payload = {"downloads": availableproducts, "label": label}

download_request_results = sendRequest(
    serviceUrl + "download-request", download_req_payload, apiKey
)
download_request_results

In [ ]:
out_dir = os.path.join(os.getcwd(), "downloads")
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

In [ ]:
# Attempt to download URLs if available
if len(download_request_results["availableDownloads"]) > 0:
    downloadrequestIDs = [
        (did.get("displayId"), did.get("entityId"))
        for did in download_request_results.get("availableDownloads", [])
        if isinstance(did, dict)
    ]
    print(f"  > Getting available downloads: {downloadrequestIDs}")
    for result in download_request_results["availableDownloads"]:
        runDownload(threads, result, out_dir)

# Get items labeled as being prepared for Download
if len(download_request_results["preparingDownloads"]) > 0:
    print(
        "  > Using download-retrieve to get download urls listed under preparingDownloads..."
    )
    print(
        f"  > {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} Pausing for 5 minutes..."
    )

    time.sleep(300)  # Wait for 300 seconds
    preparingDownloadIds = []
    for result in download_request_results["preparingDownloads"]:
        preparingDownloadIds.append(result["downloadId"])

    download_retrieve_payload = {"label": label}
    moreDownloadUrls = sendRequest(
        serviceUrl + "download-retrieve", download_retrieve_payload, apiKey, False
    )
    moreDownloadsIDs = [
        (did.get("displayId"), did.get("entityId"))
        for did in moreDownloadUrls.get("availableDownloads", [])
        if isinstance(did, dict)
    ]

    if moreDownloadUrls["available"]:
        print(
            "  > Downloading from newly available download urls from download-retrieve..."
        )
        print(
            f"  > Display and Entity IDs available from download-retrieve: {moreDownloadsIDs}"
        )
        for result in moreDownloadUrls["available"]:
            if (
                str(result["downloadId"]) in preparingDownloadIds
                or str(result["downloadId"]) in download_request_results["newRecords"]
                or str(result["downloadId"])
                in download_request_results["duplicateProducts"]
            ):
                runDownload(threads, result, out_dir)

    if moreDownloadUrls["requested"]:
        print("  > Downloading from requested downloads from download-retrieve...")
        print(
            f"  > Display and Entity IDs available from download-retrieve: {moreDownloadsIDs}"
        )
        for result in moreDownloadUrls["requested"]:
            if (
                str(result["downloadId"]) in preparingDownloadIds
                or str(result["downloadId"]) in download_request_results["newRecords"]
                or str(result["downloadId"])
                in download_request_results["duplicateProducts"]
            ):
                runDownload(threads, result, out_dir)

for thread in threads:
    thread.join()

In [ ]:
time.sleep(60)
for i, attempt_url in enumerate(ATTEMPT_DICT):
    if ATTEMPT_DICT[attempt_url]["complete"] == True:
        print(
            i,
            "Removing from download queue:",
            ATTEMPT_DICT[attempt_url]["downloadId"],
            ATTEMPT_DICT[attempt_url]["filename"],
        )
        sendRequest(
            serviceUrl + "download-remove",
            {"downloadId": ATTEMPT_DICT[attempt_url]["downloadId"]},
            apiKey,
        )

In [ ]:
endpoint = "logout"
if sendRequest(serviceUrl + endpoint, None, apiKey) == None:
    print("\n\nLogged Out\n")
else:
    print("\n\nLogout Failed\n")